In [0]:
# Spark Session
spark

In [0]:
default_db = spark.sql(
    "SELECT current_database()"
)
display(default_db)

In [0]:
%sql
show databases

In [0]:
# Read Sales parquet data
df_sales = spark.read.parquet("/data/input/sales_data.parquet")

In [0]:
# View data
display(df_sales)

In [0]:
# Write data as hive table
df_sales.write.format("parquet").mode("overwrite").option("path", "/data/output/sales_parquet_1/").saveAsTable("sales_parquet")

In [0]:
# View files
display(dbutils.fs.ls("/data/output/sales_parquet_1/"))

In [0]:
%sql
show tables in default

In [0]:
%sql

describe extended sales_parquet


In [0]:
%sql
update default.sales_parquet set amount = 0 where trx_id = '1734117021'

In [0]:
# Write Sales data as Delta Table
df_sales.write.format("delta").mode("overwrite").option("path", "/data/output/sales_delta_1/").saveAsTable("sales_delta")

In [0]:
%sql
show tables in default


In [0]:
%sql
describe extended sales_delta

In [0]:
# View files for delta
display(dbutils.fs.ls("/data/output/sales_delta_1/"))

In [0]:
dbutils.fs.head("/data/output/sales_delta_1/_delta_log/00000000000000000001.json")

In [0]:
%sql
describe history sales_delta

In [0]:
%sql
update default.sales_delta set amount = 0 where trx_id = '1734117021'


In [0]:
%sql
select * from default.sales_delta

In [0]:
# Read a particular version - pyspark api
df_sales_delta = spark.read.table("sales_delta@v1")

display(df_sales_delta.where("trx_id = '1734117021'"))

In [0]:
%sql

select * from sales_delta@v0 where trx_id = '1734117021'


In [0]:
df_new = spark.sql("select *, current_timestamp() as time_now from sales_delta@v0 where trx_id = '1734117021'")

display(df_new)

In [0]:
# Append data to existing delta table
df_new.write.format("delta").mode("append").option("mergeSchema", True).option("path", "/data/output/sales_delta_1/").saveAsTable("sales_delta")

In [0]:
# View data


In [0]:
# Reading Delta Table using Delta libraries
from delta import DeltaTable

dt = DeltaTable.forName(spark, "sales_delta")

display(dt.history())

In [0]:
# Converting a Parquet to Delta - Check and Convert

DeltaTable.isDeltaTable(spark, "/data/output/sales_delta_1")

DeltaTable.convertToDelta(spark, "parquet.`/data/output/sales_parquet_1`")

In [0]:
# Validate
display(dbutils.fs.ls("/data/output/sales_parquet_1"))

In [0]:
%sql

describe extended sales_parquet

In [0]:
%sql

CONVERT TO DELTA sales_parquet


In [0]:
%sql
RESTORE TABLE sales_delta TO VERSION AS OF 1

In [0]:
%sql

select * from sales_delta where trx_id = '1734117021'

In [0]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled","false")

dt = DeltaTable.forName(spark, "sales_delta")

dt.vacuum(0)

In [0]:
display(dbutils.fs.ls("/data/output/sales_delta_1"))

In [0]:
%sql

select * from sales_delta@v0 where trx_id = '1734117021'

In [0]:
# Converting back to Parquet
